# Selecting and Filtering Rows

*Choose columns, filter records, and define the order and size of a SQL result.*


## Prepare the Sales Table

The sales table holds five prepared records in an in-memory
SQLite database. `sale_id` is a primary key, and the other fields store
the product, category, quantity, whole-dollar unit price, and month number.
`execute()` runs one SQL statement; `executemany()` binds each prepared
tuple to the `?` placeholders. `commit()` completes the insert transaction.


In [1]:
import sqlite3

connection = sqlite3.connect(":memory:")

connection.execute("""
CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price INTEGER NOT NULL,
    month INTEGER NOT NULL
)
""")

sales_rows = [
    ("Mouse", "Accessory", 3, 25, 1),
    ("Keyboard", "Accessory", 2, 45, 1),
    ("Monitor", "Display", 1, 180, 2),
    ("Laptop Stand", "Accessory", 4, 30, 2),
    ("Tablet", "Device", 2, 220, 3),
]

connection.executemany("""
INSERT INTO sales (product_name, category, quantity, unit_price, month)
VALUES (?, ?, ?, ?, ?)
""", sales_rows)
connection.commit()

print("Prepared sales:")
for row in connection.execute("SELECT * FROM sales ORDER BY sale_id"):
    print(row)


Prepared sales:
(1, 'Mouse', 'Accessory', 3, 25, 1)
(2, 'Keyboard', 'Accessory', 2, 45, 1)
(3, 'Monitor', 'Display', 1, 180, 2)
(4, 'Laptop Stand', 'Accessory', 4, 30, 2)
(5, 'Tablet', 'Device', 2, 220, 3)


There are five sales, numbered 1 through 5. Mouse, Keyboard, and
Laptop Stand belong to Accessory; Monitor belongs to Display,
and Tablet belongs to Device.


## Select Columns and Calculate a Field

`SELECT` lists the fields in the result and `FROM` chooses the table.
Expressions can create calculated output columns. `AS` gives an expression
a readable label; it does not add a stored column to the table.


In [2]:
print("Product, quantity, and revenue:")
for row in connection.execute("""
SELECT product_name, quantity, quantity * unit_price AS revenue
FROM sales
ORDER BY sale_id
"""):
    print(row)


Product, quantity, and revenue:
('Mouse', 3, 75)
('Keyboard', 2, 90)
('Monitor', 1, 180)
('Laptop Stand', 4, 120)
('Tablet', 2, 440)


The five revenues are 75, 90, 180, 120, and 440 dollars. The table still stores quantity and unit price separately.


## Filter with `WHERE` and Sort with `ORDER BY`

`WHERE` keeps rows whose condition is true. `ORDER BY unit_price DESC`
sorts from largest to smallest; the product name breaks any price ties.
Without `ORDER BY`, SQL does not guarantee a result order.
The one-item tuple `(minimum_price,)` supplies the placeholder value.


In [3]:
minimum_price = 50
query = """
SELECT product_name, unit_price
FROM sales
WHERE unit_price >= ?
ORDER BY unit_price DESC, product_name ASC
"""

print("Products meeting the price threshold:")
for row in connection.execute(query, (minimum_price,)):
    print(row)


Products meeting the price threshold:
('Tablet', 220)
('Monitor', 180)


Tablet (220) appears before Monitor (180). All three accessories cost less than 50.


## Combine Conditions and Limit the Result

SQL uses `AND`, `OR`, and `NOT` to combine conditions. `AND` requires
both conditions here: the category is Accessory and the quantity is at
least 2. `LIMIT` keeps a specified number of rows after sorting.
`LIMIT 2` returns the two highest-quantity qualifying accessories.


In [4]:
print("Top two accessories with quantity at least 2:")
for row in connection.execute("""
SELECT product_name, quantity
FROM sales
WHERE category = ? AND quantity >= ?
ORDER BY quantity DESC, product_name ASC
LIMIT 2
""", ("Accessory", 2)):
    print(row)


Top two accessories with quantity at least 2:
('Laptop Stand', 4)
('Mouse', 3)


Laptop Stand has quantity 4 and Mouse has quantity 3. Keyboard also meets
both conditions with quantity 2, but sorting places it third and `LIMIT 2`
leaves it out of the result. All three records remain in the stored table.


## Close the Connection

`.close()` releases the database connection. This database lives only
in memory, so its tables disappear when the connection closes.


In [5]:
connection.close()
print("Connection closed.")


Connection closed.


Select the fields you need, bind filter values, and use explicit sorting before limiting the result. Keep stored columns separate from calculated query output.

[SQLite SELECT](https://www.sqlite.org/lang_select.html)
